In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.5 Check GPU Availability
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print("GPU available:", gpus)

if not gpus:
    print("\n" + "="*80)
    print("WARNING: No GPU detected! Training will be extremely slow.")
    print("Please go to Runtime > Change runtime type > select T4 GPU (or better).")
    print("Then restart the runtime and re-run all cells from the top.")
    print("="*80 + "\n")

In [ ]:
# 2. Unzip dataset to fast local Colab storage
import zipfile
import os

# Update this path if you upload the zip into a specific Drive subfolder!
zip_path = '/content/drive/MyDrive/dr_dataset.zip'
extract_path = '/content/data/augmented'
os.makedirs(extract_path, exist_ok=True)

print("Unzipping dataset (this may take a minute)...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Unzipping complete!")

In [ ]:
# 3. Setup configuration matching src/config.py
from pathlib import Path

# Paths to the unzipped Colab directory
DATA_AUG_DIR = Path('/content/data/augmented')
CHECKPOINT_DIR = Path('/content/checkpoints')
LOG_DIR = Path('/content/logs')
HISTORY_DIR = Path('/content/reports')

# Training hyperparameters
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_FROZEN = 30
EPOCHS_FINETUNE = 20
RANDOM_STATE = 42

# ImageNet normalisation
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

In [ ]:
# 4. Copy training and model logic from the repo
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

def _preprocess_fn(image):
    image = tf.cast(image, tf.float32) / 255.0
    mean = tf.constant(IMAGENET_MEAN, dtype=tf.float32)
    std  = tf.constant(IMAGENET_STD,  dtype=tf.float32)
    return (image - mean) / std

# Load Data Generators
datagen = ImageDataGenerator(preprocessing_function=_preprocess_fn)
train_gen = datagen.flow_from_directory(
    str(DATA_AUG_DIR / "train"),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=True,
    seed=RANDOM_STATE
)
val_gen = datagen.flow_from_directory(
    str(DATA_AUG_DIR / "val"),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="sparse",
    shuffle=False,
    seed=RANDOM_STATE
)

# Extract mappings and compute class weights
print(f"Detected class mapping: {train_gen.class_indices}")
classes = np.unique(train_gen.classes)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_gen.classes)
class_weights = dict(zip(classes.tolist(), weights.tolist()))
print("Class weights:", {k: round(v, 4) for k, v in class_weights.items()})

def build_model(freeze_base=True):
    base_model = EfficientNetB3(include_top=False, weights="imagenet", input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
    base_model.trainable = not freeze_base

    inputs = keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
    x = base_model(inputs, training=not freeze_base)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(5, activation="softmax")(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

def unfreeze_top_layers(model, n_layers=30):
    base_model = model.get_layer("efficientnetb3")
    base_model.trainable = True
    for layer in base_model.layers[:-n_layers]:
        layer.trainable = False
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
# 5. Run the Two-Stage Training Pipeline
import json
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
HISTORY_DIR.mkdir(parents=True, exist_ok=True)

# ── Stage 1: Frozen Base ──
print("\n── Stage 1: Training classification head (frozen base) ──")
callbacks_1 = [
    ModelCheckpoint(filepath=str(CHECKPOINT_DIR / "best_model.keras"), monitor="val_accuracy", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-7, verbose=1),
    CSVLogger(str(LOG_DIR / "training_log_stage1.csv"))
]

model = build_model(freeze_base=True)
history_1 = model.fit(
    train_gen, epochs=EPOCHS_FROZEN, validation_data=val_gen, class_weight=class_weights, callbacks=callbacks_1
)

with open(HISTORY_DIR / "stage1_history.json", "w") as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history_1.history.items()}, f)

# ── Stage 2: Fine-Tuning ──
print("\n── Stage 2: Fine-tuning top layers of EfficientNetB3 ──")
callbacks_2 = [
    ModelCheckpoint(filepath=str(CHECKPOINT_DIR / "finetune_best_model.keras"), monitor="val_accuracy", save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-7, verbose=1),
    CSVLogger(str(LOG_DIR / "training_log_stage2.csv"))
]

model = unfreeze_top_layers(model, n_layers=30)
history_2 = model.fit(
    train_gen, epochs=EPOCHS_FINETUNE, validation_data=val_gen, class_weight=class_weights, callbacks=callbacks_2
)

with open(HISTORY_DIR / "stage2_history.json", "w") as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history_2.history.items()}, f)

In [ ]:
# 6. Copy final best model checkpoints and history back to Google Drive
import shutil

drive_root = Path('/content/drive/MyDrive')

# Files to copy to drive
files_to_export = [
    CHECKPOINT_DIR / 'best_model.keras',
    CHECKPOINT_DIR / 'finetune_best_model.keras',
    HISTORY_DIR / 'stage1_history.json',
    HISTORY_DIR / 'stage2_history.json'
]

print("Exporting files to Google Drive...")
for local_path in files_to_export:
    if local_path.exists():
        drive_save_path = drive_root / local_path.name
        shutil.copy2(str(local_path), str(drive_save_path))
        print(f"Copied: {local_path.name}")
    else:
        print(f"File not found, skipped: {local_path.name}")

print("\n" + "="*50)
print("TRAINING SUMMARY")
print("="*50)
best_val_acc_1 = max(history_1.history.get('val_accuracy', [0]))
best_val_acc_2 = max(history_2.history.get('val_accuracy', [0]))

print(f"Stage 1 (Frozen) Best Val Accuracy:     {best_val_acc_1:.4f}")
print(f"Stage 2 (Fine-Tuned) Best Val Accuracy: {best_val_acc_2:.4f}")